In [6]:
import pandas as pd
import numpy as np
import xgboost as xgb
import gc
from itertools import product
import warnings

# 경고 메시지 억제 (깔끔한 출력을 위함)
warnings.filterwarnings('ignore')

# ==========================================
# 1. 설정 및 하이퍼파라미터 정의
# ==========================================
class Config:
    # 수정된 계획의 핵심: 최대 시차를 12로 제한하여 학습 데이터 확보 및 계절성 반영
    MAX_LAG = 12 
    # 생성할 시차 변수 목록: 단기(1,2,3), 반기(6), 연간(12)
    LAGS = [1, 2, 3, 6, 12]
    # 이동 평균 및 표준편차 윈도우 크기
    ROLLING_WINDOWS = [3, 6, 12]
    # XGBoost 하이퍼파라미터 (회귀 및 시계열 특화)
    # XGB_PARAMS = {
    #     'n_estimators': 3000,          # 충분한 트리 수
    #     'learning_rate': 0.01,         # 과적합 방지를 위한 낮은 학습률
    #     'max_depth': 6,                # 복잡한 상호작용 포착
    #     'subsample': 0.8,              # 행 샘플링
    #     'colsample_bytree': 0.8,       # 열 샘플링
    #     'objective': 'reg:squarederror', # 회귀 목적 함수
    #     'n_jobs': -1,                  # 병렬 처리
    #     'random_state': 42,
    #     'early_stopping_rounds': 100,  # 조기 종료
    #     'tree_method': 'hist'          # 대용량 데이터 고속 처리
    # }
    XGB_PARAMS = {
        'n_estimators': 3000,          # 학습 횟수 늘림
        'learning_rate': 0.01,         # 학습률 더 낮춤
        'max_depth': 6,                # [수정] 깊이 줄임 (과적합 방지)
        'subsample': 0.7,
        'colsample_bytree': 0.6,       # [수정] 피처 샘플링 비율 줄임
        'reg_alpha': 0.1,              # [추가] L1 규제 (불필요한 특성 무시)
        'reg_lambda': 1.0,             # [추가] L2 규제 (가중치 억제)
        'objective': 'reg:squarederror',
        'n_jobs': -1,
        'random_state': 42,
        'tree_method': 'hist' 
    }

# ==========================================
# 2. 데이터 로드 및 구조 분석 함수
# ==========================================
def load_data():
    """
    데이터를 로드하고 기본 정보를 출력합니다.
    """
    print(">>> 데이터 로딩 중...")
    try:
        # : 학습 데이터
        train = pd.read_csv('../data/train.csv')
        # : 제출 양식 (예측해야 할 쌍 정보 포함)
        test_structure = pd.read_csv('../results/sample_submission.csv')
        print(f"Train shape: {train.shape}, Submission shape: {test_structure.shape}")
        return train, test_structure
    except FileNotFoundError:
        print("오류: 'train.csv' 또는 'sample_submission.csv' 파일을 찾을 수 없습니다.")
        return None, None

# ==========================================
# 3. 전처리 및 시계열 구조화
# ==========================================
def preprocess_data(df):
    """
    연월 정보를 연속적인 인덱스(date_block_num)로 변환하고
    월별 아이템 총액으로 집계합니다.
    """
    print(">>> 데이터 전처리 중...")
    
    # 2022년 1월을 0으로 하는 연속적 시간 인덱스 생성
    # 2022-01 -> 0, 2023-01 -> 12, 2024-01 -> 24
    df['date_block_num'] = (df['year'] - 2022) * 12 + (df['month'] - 1)
    
    # 월별, 아이템별 집계 (중복 seq 제거 및 총합 계산)
    # Target Variable: 'value'
    df_grouped = df.groupby(['item_id', 'date_block_num'])['value'].sum().reset_index()
    
    return df_grouped

def create_full_grid(df, target_items):
    """
    모든 아이템과 모든 시간의 조합(Cartesian Product)을 생성하여
    거래가 없는 달(결측치)을 0으로 채웁니다. (Sparsity 해결)
    """
    print(">>> 전체 시계열 그리드(Grid) 생성 중...")
    
    min_date = df['date_block_num'].min()
    max_date = df['date_block_num'].max()
    all_dates = np.arange(min_date, max_date + 1)
    
    # 그리드 생성: 모든 날짜 x 타겟 아이템
    # 메모리 효율을 위해 numpy 배열 활용
    grid = []
    for date in all_dates:
        curr_items = target_items # 전체 기간에 대해 모든 아이템 고려
        grid.append(np.array(list(product([date], curr_items))))
    
    grid = pd.DataFrame(np.vstack(grid), columns=['date_block_num', 'item_id'])

    # [추가] NumPy가 날짜를 문자로 바꿨을 수 있으므로, 다시 숫자로 강제 변환
    grid['date_block_num'] = grid['date_block_num'].astype(int)
    
    # 원본 데이터와 병합 (Left Join)
    all_data = pd.merge(grid, df, on=['date_block_num', 'item_id'], how='left')
    
    # 결측치(거래 없음)는 0으로 채움
    all_data['value'] = all_data['value'].fillna(0)
    
    # Value에 로그 변환 적용 (데이터 분포 안정화, NMAE 최적화)
    # log(1+x) 변환
    all_data['value_log'] = np.log1p(all_data['value'])
    
    return all_data

# ==========================================
# 4. 특성 공학 (Feature Engineering): Lag 12 전략
# ==========================================
def generate_lag_features(df):
    """
    설정된 Config.LAGS와 Config.ROLLING_WINDOWS에 따라
    파생 변수를 생성합니다. (수정된 계획의 핵심 구현)
    """
    print(f">>> 특성 생성 중 (Max Lag: {Config.MAX_LAG})...")
    
    # 아이템별, 시간순 정렬
    df = df.sort_values(['item_id', 'date_block_num'])
    
    # 1. 시차 변수 (Lag Features) 생성
    for lag in Config.LAGS:
        # 그룹별 shift 연산으로 과거 값 가져오기
        df[f'value_lag_{lag}'] = df.groupby('item_id')['value_log'].shift(lag)
        
    # 2. 이동 통계 변수 (Rolling Features) 생성
    # 주의: 현재 달의 정보를 쓰지 않기 위해 lag_1을 기준으로 rolling 수행
    for win in Config.ROLLING_WINDOWS:
        # 이동 평균
        df[f'rolling_mean_{win}'] = df.groupby('item_id')['value_lag_1'].transform(
            lambda x: x.rolling(window=win).mean()
        )
        # 이동 표준편차 (변동성)
        df[f'rolling_std_{win}'] = df.groupby('item_id')['value_lag_1'].transform(
            lambda x: x.rolling(window=win).std()
        )
        
    return df

# ==========================================
# 5. 모델 학습 및 예측 파이프라인
# ==========================================
def run_pipeline():
    # 1. 데이터 로드
    train_raw, submission_struct = load_data()
    if train_raw is None: return

    # 2. 전처리
    df_clean = preprocess_data(train_raw)
    
    # 제출 파일에 등장하는 모든 아이템 식별 (Leading + Following)
    # 이 아이템들에 대해서만 시계열 특성을 만들면 메모리를 절약할 수 있음
    relevant_items = set(submission_struct['leading_item_id']).union(set(submission_struct['following_item_id']))
    df_clean = df_clean[df_clean['item_id'].isin(relevant_items)]
    
    # 3. 그리드 생성 및 특성 공학
    # 예측 대상 시점(Target Date)을 파악
    # 학습 데이터의 마지막 달 + 1을 예측해야 함
    max_train_date = df_clean['date_block_num'].max()
    target_date = max_train_date + 1
    print(f"학습 데이터 마지막 시점: {max_train_date}, 예측 대상 시점: {target_date}")
    
    # 특성 생성을 위해 타겟 시점까지 포함된 그리드 생성 (값은 NaN)
    # 2025-07(예상) 데이터 행을 미리 만들어두고 shift를 통해 채워지게 함
    all_data = create_full_grid(df_clean, list(relevant_items))
    
    # 타겟 시점의 더미 행 추가 (특성 계산용)
    target_grid = pd.DataFrame({
        'date_block_num': [target_date] * len(relevant_items),
        'item_id': list(relevant_items),
        'value': [0] * len(relevant_items), # 실제값 모름, 0으로 채움
        'value_log': [np.nan] * len(relevant_items) # 로그값 NaN으로 두어 구분
    })
    
    # 전체 데이터 합치기
    full_data = pd.concat([all_data, target_grid], axis=0, ignore_index=True)
    
    # 특성 생성 실행
    full_data = generate_lag_features(full_data)
    
    # 4. 학습 데이터셋 구성 (Pairs 구성)
    print(">>> 학습 데이터셋 병합 및 구성 중...")
    
    # 우리는 (Leading Item, Following Item) 쌍에 대해 예측해야 함.
    # 모델의 입력: Leading Item의 특성 + Following Item의 특성 -> 출력: Following Item의 다음달 Value
    
    # 데이터 분할: 학습용(과거 데이터) vs 예측용(타겟 데이터)
    # 학습용: Lag 12가 확보된 시점(12)부터 마지막 시점(max_train_date)까지
    valid_train_dates = full_data[full_data['date_block_num'] >= Config.MAX_LAG]['date_block_num'].unique()
    valid_train_dates = valid_train_dates[valid_train_dates < target_date]
    
    # 메모리 효율성을 위해 필요한 컬럼만 선택
    feature_cols = [c for c in full_data.columns if 'lag' in c or 'rolling' in c]
    
    # Leading Item 특성 테이블
    lead_feats = full_data[['date_block_num', 'item_id'] + feature_cols].copy()
    lead_feats.columns = ['date_block_num', 'leading_item_id'] + [f'lead_{c}' for c in feature_cols]
    
    # Following Item 특성 테이블 (Target 포함)
    follow_feats = full_data[['date_block_num', 'item_id', 'value_log'] + feature_cols].copy()
    follow_feats.rename(columns={'item_id': 'following_item_id', 'value_log': 'target'}, inplace=True)
    
    # 제출 파일에 있는 쌍(Pair) 정보만 가져오기
    target_pairs = submission_struct[['leading_item_id', 'following_item_id']].drop_duplicates()
    
    # 학습 데이터 생성 (Cross Join of Pairs * Valid Dates)
    # 데이터가 클 수 있으므로 반복문으로 처리하거나 메모리 관리 필요
    # 여기서는 간소화를 위해 pandas merge 활용
    
    # 날짜별로 쌍 데이터를 복제
    train_pairs_list = []
    for date in valid_train_dates:
        temp = target_pairs.copy()
        temp['date_block_num'] = date
        train_pairs_list.append(temp)
    train_base = pd.concat(train_pairs_list, ignore_index=True)
    
    # Leading Features 병합
    train_df = pd.merge(train_base, lead_feats, on=['date_block_num', 'leading_item_id'], how='left')
    # Following Features 병합
    train_df = pd.merge(train_df, follow_feats, on=['date_block_num', 'following_item_id'], how='left')
    
    # 결측치 제거 (Lag로 인해 앞부분 NaN 발생)
    train_df = train_df.dropna()
    
    # 5. 모델 학습 (XGBoost)
    print(">>> 모델 학습 시작 (XGBoost)...")
    
    # 특성 및 타겟 분리
    X_cols = [c for c in train_df.columns if c not in ['leading_item_id', 'following_item_id', 'date_block_num', 'target']]
    X = train_df[X_cols]
    y = train_df['target']
    
    # 검증셋 분리 (마지막 3개월)
    val_start_date = max_train_date - 2
    is_val = train_df['date_block_num'] >= val_start_date
    
    X_train, y_train = X[~is_val], y[~is_val]
    X_val, y_val = X[is_val], y[is_val]
    
    model = xgb.XGBRegressor(**Config.XGB_PARAMS)
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=100
    )
    
    # 6. 최종 예측 (Inference)
    print(">>> 최종 예측 데이터 생성 중...")
    
    # 예측용 베이스 생성 (타겟 날짜)
    test_base = target_pairs.copy()
    test_base['date_block_num'] = target_date
    
    # 타겟 날짜의 특성 데이터 가져오기
    # Leading Items (타겟 날짜의 Lag들은 이미 계산되어 있음)
    lead_target = lead_feats[lead_feats['date_block_num'] == target_date]
    # Following Items
    follow_target = follow_feats[follow_feats['date_block_num'] == target_date].drop(columns=['target'])
    
    # 병합
    test_df = pd.merge(test_base, lead_target, on=['date_block_num', 'leading_item_id'], how='left')
    test_df = pd.merge(test_df, follow_target, on=['date_block_num', 'following_item_id'], how='left')
    
    # 결측치 처리 (혹시 모를 누락 대비)
    test_df = test_df.fillna(0)
    
    # 예측 수행
    pred_log = model.predict(test_df[X_cols])
    
    # 역변환 (Log -> Expm1)
    pred_value = np.expm1(pred_log)
    
    # 음수 값 처리 (무역량은 음수 불가)
    pred_value = np.maximum(pred_value, 0)
    
    # 7. 제출 파일 구조 검증 및 생성
    print(">>> 제출 파일 생성 및 구조 검증...")
    
    submission = submission_struct.copy()
    
    # 결과를 submission 구조에 매핑
    # 순서가 섞이지 않도록 merge를 사용하고 value 컬럼 업데이트
    test_df['pred_value'] = pred_value
    
    final_sub = pd.merge(submission[['leading_item_id', 'following_item_id']], 
                         test_df[['leading_item_id', 'following_item_id', 'pred_value']],
                         on=['leading_item_id', 'following_item_id'], 
                         how='left')
    
    # 결측치 확인 (매핑 실패 시 0 처리)
    n_missing = final_sub['pred_value'].isnull().sum()
    if n_missing > 0:
        print(f"경고: {n_missing}개의 쌍에 대한 예측값이 누락되었습니다. 0으로 대체합니다.")
        final_sub['pred_value'] = final_sub['pred_value'].fillna(0)
        
    # 컬럼명 변경 및 저장
    submission['value'] = final_sub['pred_value']
    
    # 최종 파일 저장
    submission.to_csv('final_submission_lag12_optimized.csv', index=False)
    print(">>> 완료: 'final_submission_lag12_optimized.csv' 저장됨.")

if __name__ == "__main__":
    run_pipeline()

>>> 데이터 로딩 중...
Train shape: (10836, 9), Submission shape: (9900, 3)
>>> 데이터 전처리 중...
학습 데이터 마지막 시점: 42, 예측 대상 시점: 43
>>> 전체 시계열 그리드(Grid) 생성 중...
>>> 특성 생성 중 (Max Lag: 12)...
>>> 학습 데이터셋 병합 및 구성 중...
>>> 모델 학습 시작 (XGBoost)...
[0]	validation_0-rmse:4.94308	validation_1-rmse:4.81337
[100]	validation_0-rmse:2.42008	validation_1-rmse:2.70050
[200]	validation_0-rmse:1.69102	validation_1-rmse:2.25488
[300]	validation_0-rmse:1.44378	validation_1-rmse:2.18467
[400]	validation_0-rmse:1.29250	validation_1-rmse:2.19678
[500]	validation_0-rmse:1.19147	validation_1-rmse:2.20568
[600]	validation_0-rmse:1.11511	validation_1-rmse:2.21186
[700]	validation_0-rmse:1.04857	validation_1-rmse:2.22184
[800]	validation_0-rmse:0.99109	validation_1-rmse:2.23424
[900]	validation_0-rmse:0.94034	validation_1-rmse:2.24506
[1000]	validation_0-rmse:0.89406	validation_1-rmse:2.25440
[1100]	validation_0-rmse:0.85261	validation_1-rmse:2.26044
[1200]	validation_0-rmse:0.81639	validation_1-rmse:2.26675
[1300]	validation_